# 04. Contact events

Mean observed velocity in the 1 s and 2 s before each gripper contact. Run after `02_run_trial`.

| | |
|---|---|
| Input | `result/final_N.xlsx`, contact frames read from `video_N.mp4` |
| Output | `result/contact_N.csv` (contact frames), Contact_Events sheet in `final_N.xlsx` |
| GPU | Not required |

For each trial: cell 1 (trial number), cell 2 (enter contact frames and click **Save**), cell 3 (compute).
The averaging window ends 5 frames before contact so that the velocity does not use positions after contact.

In [ ]:
#@title 0. Setup: repository, Drive, day folder, day settings
REPO_URL = "https://github.com/dodokim777-boop/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
from main_code import settings
from main_code.common.notebook_ui import ask_day_folder, ask_trial, day_settings

DAY = ask_day_folder()
SETTINGS_FILE = day_settings(DAY, os.path.join(REPO_DIR, 'experiment_settings.yaml'))

In [ ]:
#@title 1. Trial number
TRIAL = ask_trial(DAY)
settings.configure(settings.day_settings_path(DAY), data_root=DAY, trial=TRIAL)
if not os.path.exists(settings.FINAL_EXCEL):
    raise RuntimeError(f'{settings.FINAL_EXCEL} not found. Run 02_run_trial for trial {TRIAL} first.')
_frames = pd.read_excel(settings.FINAL_EXCEL, sheet_name='Result', usecols=['Frame'])['Frame']
FRAME_RANGE = (int(_frames.min()), int(_frames.max()))
print(f'Trial {TRIAL}: {settings.FINAL_EXCEL} (frames {FRAME_RANGE[0]}-{FRAME_RANGE[1]})')

In [ ]:
#@title 2. Contact frames (edit, then click Save)
from main_code.common.notebook_ui import contact_form
from main_code.postprocessing.contact_events import contact_csv_path
contact_form(contact_csv_path(), FRAME_RANGE)

In [ ]:
#@title 3. Compute Contact_Events
from main_code.postprocessing.contact_events import run_contact_events
events = run_contact_events()
if events is not None:
    display(events)